**`import_parcels`**

Script examples to import parcel dataset

# Configure

In [ ]:
import argparse

from openplaces.io.ingester import Ingester
from openplaces.recipe import get_recipe_by_id
from openplaces.utils import pretty_print

In [ ]:
# Define arguments
parser = argparse.ArgumentParser(description='Ingest parcels using a recipe')
parser.add_argument(
    '--recipe_id',
    help='Identifier of the recipe (e.g., "US-NC_parcel-nconemap-2025")',
)
parser.add_argument(
    '--admin_ids',
    help='Administrative unit IDs to ingest (e.g., "US-RI")',
    nargs='*',
);

# Set arguments

In [ ]:
ARGS_TEST = (
    # Example 1: North Carolina OneMap
    # Buncombe and Brunswick counties, NC (flood risk pilots)
    '--recipe_id US-NC_parcel-nconemap-2025 '
    '--admin_ids US-NC-BO US-NC-BS'
    # Example 2: Florida Geographic Information Office
    # Hillsborough, Pinellas, and Manatee counties, FL (Tampa bay)
    # '--recipe_id US-FL_parcel-floridagio-2025 '
    # '--admin_ids US-FL-HL US-FL-MN US-FL-PI'
    # Example 3: Wisconsin Statewide Parcel data
    # Oneida and Vilas counties, WI (lake hedonics case study)
    # '--recipe_id US-WI_parcel-wiscedu-v11 '
    # '--admin_ids US-WI-ON US-WI-VI'
    # Example 4: Texas Geographical Information Office (TxGIO)
    # Jefferson and Harris counties, TX (CHEER case studies)
    # '--recipe_id US-TX_parcel-txgio-2025 '
    # '--admin_ids US-TX-JE US-TX-RR'
    # Example 5: Virginia
    # Albemarle county and Charlottesville, VA (county-city example)
    # '--recipe_id US-VA_parcel-vgin-2025 '
    # '--admin_ids US-VA-AB US-VA-CE'
)

# Convert argument string to list of strings
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse list of arguments
args = parser.parse_args(args_list)

# Display arguments to check if parsing worked as expected
args

# Show recipe

In [ ]:
pretty_print(get_recipe_by_id(args.recipe_id))

# Ingest parcel data

In [ ]:
ingester = Ingester(
    args.recipe_id,
    args.admin_ids,
    verbose=True,
)

In [ ]:
import os.path

In [ ]:
ingester.ingest()
# ingester.ingest(reprocess=True)
# ingester.ingest(redownload=True)

# Inspect results

## Show full map

In [ ]:
import contextily as cx
import matplotlib.pyplot as plt
import shapely

from openplaces.api import get_admin, read_entities
from openplaces.core.schema import AdminId

if ingester.admin_ids_to_save:

    last_saved_admin_id = ingester.admin_ids_to_save[-1]
    print(last_saved_admin_id)

    level = AdminId(last_saved_admin_id).get_level()
    admin_recipe = f'US_admin-nhgis-2020_admin{level}'
    admin = get_admin(last_saved_admin_id, level=level, recipe=admin_recipe, geom=True)

    parcels = read_entities(ingester.recipe, last_saved_admin_id, geom=True)

    fig, ax = plt.subplots(figsize=(10, 10))

    if isinstance(
        parcels.geometry.iloc[0],
        (shapely.geometry.Polygon, shapely.geometry.MultiPolygon),
    ):
        if len(parcels) > 250000:
            print('>250K parcel polygons to plot. Taking sample')
            _parcels_plot = parcels.sample(250000)
        else:
            _parcels_plot = parcels
        _parcels_plot.boundary.plot(ax=ax, color='blue', linewidth=0.3, alpha=0.5)

    admin.boundary.plot(ax=ax, color='black', linewidth=0.5)
    ax.set_title(
        f'{len(parcels):,d} parcels in '
        + admin.loc[last_saved_admin_id, 'name']
        + f' ({last_saved_admin_id}) from `{args.recipe_id}`'
    )
    ax.axis('off')
    cx.add_basemap(
        ax,
        crs=parcels.crs,
        source=cx.providers.Esri.WorldImagery,
        alpha=0.5,
    )

## Show random parcel with attributes

In [ ]:
from openplaces.viz import show_polygon_context
if ingester.admin_ids_to_save:
    show_polygon_context(parcels, parcels.sample().index[0]);